# Fine-tune for entity parity — explicit condition

Trains `Qwen3-4B-Instruct-2507` on the `parity_answer` column so that lowering the register
stops removing substantive content.

## Algorithm

**QLoRA / LoRA supervised fine-tuning (SFT) with completion-only loss masking.**

* **LoRA** (low-rank adaptation) — the base weights stay frozen and small rank-16 adapter
  matrices are trained on the attention and MLP projections. ~33M trainable parameters out
  of 4B, which is what makes this fit on a 16 GB card, and it makes the change reversible:
  the "before" model is the same file with the adapter detached, so the before/after
  comparison is exact rather than two separately-trained models.
* **Completion-only masking** — loss is computed on the assistant turn only. The persona
  template is identical across two thirds of the dataset; without masking it would dominate
  the loss and the model would learn to echo prompts.
* **Contrastive triplets** — each event contributes all three literacy levels with the same
  topic, so what is learned is the *mapping* from literacy signal to register, holding
  content fixed.

### Why SFT and not DPO/ORPO

You do have natural preference pairs here — `cleaned_answer` (drops entities) is a ready-made
*rejected* for every `parity_answer` *chosen*. Preference optimisation is the obvious next
step and ORPO needs no reference model, so it fits the same GPU. It is not the right first
move: SFT on 339 examples is stable, has one loss term to diagnose, and produces a result
that is trivial to describe in a paper. Establish that baseline, then try ORPO as an ablation.

## What the training data encodes

| Paper metric | Direction wanted | How the data carries it |
|---|---|---|
| Specific entity count | preserve | Mid/Low targets keep High's entities (filtered to ≥0.85 coverage) |
| NLI entailment | preserve | targets are rewrites of High, so propositions are shared |
| Semantic similarity | preserve | same |
| Info axis (comprehensive) | preserve | targets stay detailed |
| **FK grade** | **still descend** | targets keep the register descent (14.5 → 8.2 → 6.5) |
| **Style axis (formal)** | **still descend** | Low targets are deliberately conversational |

The High rows are in the training set with their **own unchanged responses**. Training only
on Mid/Low would teach the model to write simply for everyone and destroy the High-literacy
behaviour the audit measures against.

In [1]:
from pathlib import Path
import os, sys, json, math, random, time

import numpy as np
import torch
from torch.utils.data import Dataset
from transformers import (AutoModelForCausalLM, AutoTokenizer, Trainer,
                          TrainingArguments, set_seed)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from train_parity import (load_rows, group_by_event, split_event_ids, build_records,
                          mask_labels, encode_pair, make_training_args,
                          save_split, entity_coverage,
                          LEVELS, ANCHOR_LEVEL, IGNORE_INDEX)

DATA   = ROOT / "newly_generated_answers" / "explicit_answers_local_base.csv"
OUTDIR = ROOT / "models" / "qwen4b-parity-explicit"

print("root  :", ROOT)
print("data  :", DATA.name, "|", "found" if DATA.exists() else "MISSING")
print("torch :", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print("gpu   :", p.name, f"| {p.total_memory/1024**3:.1f} GiB")

/home/anirban/Desktop/Research/qwen_before_after_audit_package_fixed_v2/qwen_before_after_audit/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


root  : /home/anirban/Desktop/Research/qwen_before_after_audit_package_fixed_v2/qwen_before_after_audit
data  : explicit_answers_local_base.csv | found
torch : 2.14.0+cu130 | cuda: True
gpu   : NVIDIA GeForce RTX 5060 Ti | 15.5 GiB


## 1. Configuration

**Memory.** bf16 LoRA on this 4B is roughly 8.0 GB weights + 0.4 GB adapter/optimizer +
~1.5 GB activations with gradient checkpointing ≈ 10 GB, which fits 16 GB with headroom. That
is preferable to 4-bit for training quality; flip `LOAD_IN_4BIT` only if you hit OOM.

**`MAX_LENGTH = 2048`** — measured, not guessed. The longest prompt+target in this dataset is
~1280 tokens, so nothing truncates. Truncation would silently cut entities off the end of a
target and train the exact behaviour being removed.

**`MIN_COVERAGE = 0.85`** — the rewrite quality is bimodal (median 1.00, p10 0.17). At 0.85
the surviving targets average 0.99 coverage with no loss of register descent. Lower it to 0.70
for ~10% more data at 0.97 coverage.

In [2]:
BASE_MODEL   = "Qwen/Qwen3-4B-Instruct-2507"
LOAD_IN_4BIT = False        # True only if bf16 OOMs

SEED       = 42
TEST_FRAC  = 0.20           # event-level, same split as audit_utils
MAX_LENGTH = 2048

TARGET_COLUMN  = "parity_answer"
MIN_COVERAGE   = 0.85       # drop Mid/Low targets that lost entities
BALANCE_ANCHOR = True       # down-sample High so the three levels are even

LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

EPOCHS          = 3
LEARNING_RATE   = 2e-4
BATCH_SIZE      = 1
GRAD_ACCUM      = 8         # effective batch 8
WARMUP_RATIO    = 0.03
WEIGHT_DECAY    = 0.0
MAX_GRAD_NORM   = 1.0
LOGGING_STEPS   = 5

set_seed(SEED)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

print(f"{BASE_MODEL} | 4bit={LOAD_IN_4BIT} | lora r{LORA_R}a{LORA_ALPHA} | "
      f"{EPOCHS} epochs | eff.batch {BATCH_SIZE*GRAD_ACCUM} | lr {LEARNING_RATE} | "
      f"target='{TARGET_COLUMN}' cov>={MIN_COVERAGE}")

Qwen/Qwen3-4B-Instruct-2507 | 4bit=False | lora r16a32 | 3 epochs | eff.batch 8 | lr 0.0002 | target='parity_answer' cov>=0.85


## 2. Build the dataset

The split is **event-level**. All three literacy levels of an event are near-paraphrases of
one another, so a row-level split would put a variant of every test target into training and
the held-out loss would be meaningless.

In [3]:
rows   = load_rows(DATA)
events = group_by_event(rows)
train_ids, test_ids = split_event_ids(events.keys(), TEST_FRAC, SEED)

train_recs, tr_stats = build_records(events, set(train_ids), TARGET_COLUMN,
                                     min_coverage=MIN_COVERAGE,
                                     balance_anchor=BALANCE_ANCHOR, seed=SEED)
eval_recs, ev_stats  = build_records(events, set(test_ids), TARGET_COLUMN,
                                     min_coverage=MIN_COVERAGE,
                                     balance_anchor=BALANCE_ANCHOR, seed=SEED)

assert not (set(train_ids) & set(test_ids)), "event leaked across the split"
assert not ({r['event_id'] for r in train_recs} & {r['event_id'] for r in eval_recs})

print(f"events      : {len(events)}  ->  train {len(train_ids)} / test {len(test_ids)}")
print(f"train recs  : {tr_stats['records']:3d}  {tr_stats['by_level']}")
print(f"eval recs   : {ev_stats['records']:3d}  {ev_stats['by_level']}")
print(f"dropped     : {tr_stats['dropped']}")

steps_per_epoch = math.ceil(len(train_recs) / (BATCH_SIZE * GRAD_ACCUM))
print(f"\noptimizer steps: {steps_per_epoch}/epoch x {EPOCHS} = {steps_per_epoch*EPOCHS}")

events      : 250  ->  train 200 / test 50
train recs  : 339  {'High Literacy': 113, 'Mid Literacy': 115, 'Low Literacy': 111}
eval recs   :  77  {'High Literacy': 26, 'Mid Literacy': 26, 'Low Literacy': 25}
dropped     : {'missing': 0, 'low_coverage': 174, 'empty': 0, 'anchor_balanced': 87}

optimizer steps: 43/epoch x 3 = 129


## 3. What one training example looks like

In [4]:
r = next(x for x in train_recs if x["level"] == "Low Literacy")
print(f"event {r['event_id']} | {r['level']}\n")
print("SYSTEM :", r["messages"][0]["content"])
print("USER   :")
for ln in r["messages"][1]["content"].splitlines():
    print("        ", ln)
print("\nTARGET (first 500 chars):")
print(r["messages"][2]["content"][:500])

high = events[r["event_id"]][ANCHOR_LEVEL]["cleaned_answer"]
print(f"\nentity coverage of this target vs its High response: "
      f"{entity_coverage(high, r['messages'][2]['content']):.2f}")

event CON_07 | Low Literacy

SYSTEM : You are a helpful assistant that adapts to the user's literacy level.
USER   :
         The topic: Weight Deception
         Additional context/details: Selling goods less than the offered weight is an offence punishable by imprisonment not exceeding 1 year, or fine not exceeding 50,000 BDT, or both
         Source/Reference: Consumer Act 2009
         Persona: I have low literacy on this topic. Explain the topic according to my level. (Low Literacy)

TARGET (first 500 chars):
Weight Deception means giving less than said. 
Example: 1 kg rice given as 800 grams. 
It is illegal. 
Section 15 of the Consumer Act 2009 says no. 
Sellers must give the weight they say. 
If they don’t, it is fraud. 
No proof of intent needed. 
Small shortfalls can be wrong if repeated. 
BCPA can check and punish. 
Jail up to 1 year. 
Fine up to 50,000 BDT. 
Or both. 
It covers rice, sugar, flour, spices. 
It protects fair value and truth. 
Sellers must follow stated weights

## 4. Load model and attach LoRA

In [5]:
tok = AutoTokenizer.from_pretrained(BASE_MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

kw = dict(dtype=torch.bfloat16, device_map="auto")
if LOAD_IN_4BIT:
    from transformers import BitsAndBytesConfig
    kw.pop("dtype")
    kw["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True)

model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, **kw)
if LOAD_IN_4BIT:
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

model.config.use_cache = False          # required with gradient checkpointing
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.enable_input_require_grads()      # lets grads flow through the checkpointed frozen base

lora = LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
                  bias="none", task_type="CAUSAL_LM", target_modules=LORA_TARGETS)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

if torch.cuda.is_available():
    print(f"vram after load: {torch.cuda.memory_allocated()/1024**3:.2f} GiB")

Loading weights: 100%|██████████| 398/398 [00:05<00:00, 76.29it/s] 


trainable params: 33,030,144 || all params: 4,055,498,240 || trainable%: 0.8145
vram after load: 7.62 GiB


## 5. Tokenisation and loss masking

`mask_labels` sets every prompt token to `-100` so the loss covers the assistant turn only.
It is unit-tested in `src/train_parity.py`, including the case where the tokenizer does not
emit a clean prefix. The next cell decodes a real example to prove the masking landed on the
right span — worth actually reading, because silent mis-masking trains the model on prompts
and looks like a normal loss curve.

In [6]:
def encode(messages):
    # encode_pair normalises apply_chat_template across transformers versions. From 4.56
    # onward it returns a BatchEncoding, and list() on that yields the dict KEYS rather than
    # token ids -- which silently drops every example as unmaskable.
    return encode_pair(tok, messages)


class ParityDataset(Dataset):
    def __init__(self, records, max_length):
        self.items, self.skipped = [], {"too_long": 0, "unmaskable": 0}
        for rec in records:
            prefix, full = encode(rec["messages"])
            if len(full) > max_length:
                self.skipped["too_long"] += 1
                continue                      # skip, never truncate: the tail holds entities
            labels = mask_labels(prefix, full)
            if labels is None:
                self.skipped["unmaskable"] += 1
                continue
            self.items.append({"input_ids": full, "labels": labels})

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        return self.items[i]


def collate(batch):
    n = max(len(b["input_ids"]) for b in batch)
    pad = tok.pad_token_id
    out = {"input_ids": [], "attention_mask": [], "labels": []}
    for b in batch:
        k = n - len(b["input_ids"])
        out["input_ids"].append(b["input_ids"] + [pad] * k)
        out["attention_mask"].append([1] * len(b["input_ids"]) + [0] * k)
        out["labels"].append(b["labels"] + [IGNORE_INDEX] * k)
    return {k: torch.tensor(v, dtype=torch.long) for k, v in out.items()}


train_ds = ParityDataset(train_recs, MAX_LENGTH)
eval_ds  = ParityDataset(eval_recs, MAX_LENGTH)
print(f"train {len(train_ds)} examples (skipped {train_ds.skipped})")
print(f"eval  {len(eval_ds)} examples (skipped {eval_ds.skipped})")

lens = sorted(len(i["input_ids"]) for i in train_ds.items)
print(f"token length: p50 {lens[len(lens)//2]}  p95 {lens[int(.95*len(lens))]}  max {lens[-1]}"
      f"  (MAX_LENGTH {MAX_LENGTH})")

n_long = train_ds.skipped["too_long"] + eval_ds.skipped["too_long"]
if n_long:
    print(f"\n!! {n_long} examples dropped for exceeding MAX_LENGTH={MAX_LENGTH}. Expected 0 on "
          f"this dataset (longest is ~1280 tokens). Raise MAX_LENGTH and re-run this cell — "
          f"training on a silently reduced dataset is the failure mode here.")
assert len(train_ds) > 0, (
    f"every example was dropped: {train_ds.skipped}. A large 'unmaskable' count means\n"
    f"apply_chat_template returned a BatchEncoding instead of token ids -- check that\n"
    f"encode() routes through train_parity.encode_pair.")

train 0 examples (skipped {'too_long': 0, 'unmaskable': 339})
eval  0 examples (skipped {'too_long': 0, 'unmaskable': 77})


IndexError: list index out of range

### Verify the mask before spending GPU time

In [ ]:
it = train_ds.items[0]
trained = [t for t, l in zip(it["input_ids"], it["labels"]) if l != IGNORE_INDEX]
masked  = [t for t, l in zip(it["input_ids"], it["labels"]) if l == IGNORE_INDEX]

print(f"tokens: {len(it['input_ids'])} total | {len(masked)} masked | {len(trained)} trained on\n")
print("MASKED (no loss) — tail of the prompt:")
print("   ...", tok.decode(masked[-90:]).replace("\n", "\n    "))
print("\nTRAINED (loss) — head of the target:")
print("   ", tok.decode(trained[:90]).replace("\n", "\n    "))

assert len(trained) > 0, "nothing to train on"
assert all(l == IGNORE_INDEX for l in it["labels"][:len(masked)]), "mask is not a clean prefix"
frac = np.mean([sum(1 for l in i["labels"] if l != IGNORE_INDEX) / len(i["labels"])
                for i in train_ds.items])
print(f"\nmean fraction of tokens carrying loss: {frac:.1%}")

## 6. Train

In [ ]:
OUTDIR.mkdir(parents=True, exist_ok=True)

args_kw = dict(
    output_dir=str(OUTDIR / "checkpoints"),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    weight_decay=WEIGHT_DECAY,
    max_grad_norm=MAX_GRAD_NORM,
    logging_steps=LOGGING_STEPS,
    save_strategy="epoch",
    save_total_limit=1,
    bf16=not LOAD_IN_4BIT,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    report_to=[],
    seed=SEED,
    remove_unused_columns=False,
    label_names=["labels"],
)
steps_per_epoch = math.ceil(len(train_ds) / (BATCH_SIZE * GRAD_ACCUM))
args, dropped = make_training_args(
    TrainingArguments, warmup_ratio=WARMUP_RATIO,
    total_steps=steps_per_epoch * EPOCHS,
    eval_strategy="epoch", **args_kw)
if dropped:
    print(f"note: this transformers version does not accept {dropped}; proceeding without.")

trainer = Trainer(model=model, args=args, train_dataset=train_ds,
                  eval_dataset=eval_ds, data_collator=collate)

t0 = time.time()
result = trainer.train()
print(f"\ntrained in {(time.time()-t0)/60:.1f} min")
print(f"final train loss: {result.training_loss:.4f}")

### Loss curve

Held-out loss rising while training loss falls is over-fitting on 339 examples — drop to
2 epochs and retrain. Both flat means the LoRA never engaged: check the trainable-parameter
count and the learning rate.

In [ ]:
hist = trainer.state.log_history
tr = [(h["step"], h["loss"]) for h in hist if "loss" in h]
ev = [(h["step"], h["eval_loss"]) for h in hist if "eval_loss" in h]

print(f"  {'step':>6}{'train':>10}{'eval':>10}")
ev_map = dict(ev)
for s, l in tr:
    print(f"  {s:>6}{l:>10.4f}{('%.4f' % ev_map[s]) if s in ev_map else '':>10}")
for s, l in ev:
    if s not in dict(tr):
        print(f"  {s:>6}{'':>10}{l:>10.4f}")

if len(ev) >= 2:
    trend = "rising -- over-fitting" if ev[-1][1] > ev[0][1] else "falling"
    print(f"\n  held-out loss {ev[0][1]:.4f} -> {ev[-1][1]:.4f} ({trend})")

## 7. Save

Saves the adapter, the tokenizer, the **event split**, and a manifest of every setting and
data statistic. The split file is the important one: the post-training evaluation has to score
the same 50 held-out events, or the comparison is contaminated by events the model trained on.

In [ ]:
model.save_pretrained(str(OUTDIR))
tok.save_pretrained(str(OUTDIR))
save_split(OUTDIR / "event_split.json", train_ids, test_ids,
           {"seed": SEED, "test_frac": TEST_FRAC, "source": DATA.name})

manifest = {
    "base_model": BASE_MODEL, "algorithm": "LoRA SFT, completion-only loss masking",
    "load_in_4bit": LOAD_IN_4BIT, "seed": SEED, "max_length": MAX_LENGTH,
    "target_column": TARGET_COLUMN, "min_coverage": MIN_COVERAGE,
    "balance_anchor": BALANCE_ANCHOR,
    "lora": {"r": LORA_R, "alpha": LORA_ALPHA, "dropout": LORA_DROPOUT,
             "targets": LORA_TARGETS},
    "optim": {"epochs": EPOCHS, "lr": LEARNING_RATE, "batch": BATCH_SIZE,
              "grad_accum": GRAD_ACCUM, "scheduler": "cosine",
              "warmup_ratio": WARMUP_RATIO},
    "data": {"train_records": len(train_ds), "eval_records": len(eval_ds),
             "by_level": tr_stats["by_level"], "dropped": tr_stats["dropped"],
             "n_train_events": len(train_ids), "n_test_events": len(test_ids)},
    "final_train_loss": float(result.training_loss),
    "eval_loss": [h["eval_loss"] for h in trainer.state.log_history if "eval_loss" in h],
}
(OUTDIR / "manifest.json").write_text(json.dumps(manifest, indent=2) + "\n")

print("saved to", os.path.relpath(OUTDIR, ROOT))
for f in sorted(OUTDIR.iterdir()):
    if f.is_file():
        print(f"  {f.name:32s} {f.stat().st_size/1024:8.1f} KB")

## 8. Sanity check on a held-out event

Not the evaluation — that comes from regenerating all 750 answers and running the paper's
metrics. This is a two-minute look at whether training moved anything, on an event the model
never saw. Read for the dissociation: entity coverage up at Mid/Low, FK still descending.

In [ ]:
model.config.use_cache = True
model.eval()

probe_id = test_ids[0]
ev_rows = events[probe_id]
print(f"held-out event {probe_id}: {ev_rows[ANCHOR_LEVEL]['Topic']}\n")
print(f"  {'level':16s}{'words':>7s}{'FK':>7s}{'cov':>7s}   before (cov / FK)")

from parity_rewrite import fk_grade
high_ref = ev_rows[ANCHOR_LEVEL]["cleaned_answer"]

for lvl in LEVELS:
    row = ev_rows[lvl]
    msgs = [{"role": "system", "content": row["System_Prompt"]},
            {"role": "user", "content": row["User_Prompt"]}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    enc = tok(text, return_tensors="pt", add_special_tokens=False).to(model.device)
    with torch.inference_mode():
        out = model.generate(**enc, max_new_tokens=1024, do_sample=True, temperature=0.7,
                             top_p=0.8, top_k=20, pad_token_id=tok.pad_token_id)
    gen = tok.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    b = row["cleaned_answer"]
    print(f"  {lvl:16s}{len(gen.split()):7d}{fk_grade(gen):7.1f}{entity_coverage(high_ref, gen):7.2f}"
          f"   {entity_coverage(high_ref, b):.2f} / {fk_grade(b):.1f}")
    if lvl == "Low Literacy":
        print(f"\n  Low output (first 400 chars):\n   {gen[:400]}")

model.config.use_cache = False

## 9. Next

1. **Regenerate all 750 explicit answers** with notebook `00_generate_answers_local_qwen.ipynb`:
   set `ADAPTER_PATH = "models/qwen4b-parity-explicit"` and `RUN_TAG = "local_finetuned"`,
   restart the kernel, run. Same prompts, new files — that is the "after" side.
2. **Score only the 50 held-out events** from `event_split.json`. The 200 training events are
   memorised and will flatter the result; reporting them as evidence would be the single
   easiest thing for a reviewer to attack.
3. **Run the paper's metrics** on before vs after: FK, word count, ROUGE-Lsum, specific entity
   count (the rubric extractor, not the regex proxy), BGE-M3 semantic loss, NLI entailment,
   and the two posture axes.
4. **Judge on the dissociation.** Success is entity count / NLI / semantic / Info axis holding
   flat across levels while FK and Style still descend. Everything moving together is a
   failure in either direction.

Then repeat for implicit — same notebook, point `DATA` at `implicit_answers_local_base.csv`,
and filter out the ~11 High responses that deny their own source instrument first.